# Technique 2

## Case 3: Lunar Powered Descent

Ten shooting strategies on the same 5000 random initial guesses: single shooting at $\epsilon=1$ and at the target level $\epsilon_f=10^{-5}$, stepwise homotopy $\epsilon=1,10^{-1},\ldots,10^{-5}$ with backtracking, the ungated augmented formulation, and augmented shooting with the exponential and rational gates for $p=1$, $2$ and $4$. The gate thresholds are those selected in Case 1 and are transferred without retuning. Convergence is judged by the physical shooting residual at the target smoothing level.

In [1]:
import os
import sys
import platform

for name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS',
             'VECLIB_MAXIMUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[name] = '1'

import numpy as np
import scipy
import heyoka as hy
from IPython.display import display, Markdown

hy.set_nthreads(1)

In [2]:
# =========================
# Timing Settings
# =========================
from collections import Counter

cpu_samples = []
fast_cpus = []
if sys.platform == 'darwin':
    import ctypes
    import plistlib
    import subprocess
    tree = plistlib.loads(subprocess.check_output(
        ['/usr/sbin/ioreg', '-a', '-l', '-p', 'IODeviceTree', '-n', 'cpus', '-r']))
    pending = list(tree)
    while pending:
        node = pending.pop()
        if node.get('cluster-type') == b'P\x00' and 'cpu-id' in node:
            fast_cpus.append(int.from_bytes(node['cpu-id'], 'little'))
        pending.extend(node.get('IORegistryEntryChildren', []))
    lib = ctypes.CDLL('/usr/lib/libSystem.B.dylib')
    lib.pthread_set_qos_class_self_np.argtypes = [ctypes.c_uint, ctypes.c_int]
    lib.pthread_cpu_number_np.argtypes = [ctypes.POINTER(ctypes.c_size_t)]

def set_timing_priority():
    if sys.platform == 'darwin':
        assert lib.pthread_set_qos_class_self_np(0x21, 0) == 0

def sample_cpu():
    if sys.platform == 'darwin':
        cpu = ctypes.c_size_t()
        assert lib.pthread_cpu_number_np(ctypes.byref(cpu)) == 0
        cpu_samples.append(cpu.value)

environment = dict(python=sys.version, executable=sys.executable,
                   platform=platform.platform(), machine=platform.machine(),
                   numpy=np.__version__, scipy=scipy.__version__, heyoka=hy.__version__,
                   threads=1, fast_cpus=sorted(fast_cpus), strict_cpu_pinning=False)
print(environment)

records_by_method = {}
results_by_method = {}

{'python': '3.12.14 (main, Sep  3 2026, 03:46:19) [Clang 21.1.8 ]', 'executable': '/Users/zhong/Documents/Codex/miniforge3/envs/gtoc13/bin/python', 'platform': 'macOS-26.5.1-arm64-arm-64bit', 'machine': 'arm64', 'numpy': '2.5.3', 'scipy': '1.18.0', 'heyoka': '7.10.1', 'threads': 1, 'fast_cpus': [12, 13, 14, 15, 16, 17], 'strict_cpu_pinning': False}


In [3]:
import numpy as np

# Cartesian state and costate equations with logarithmic smoothing.

# =========================
# Setup (SI Units)
# =========================
# Lunar model and propulsion parameters
mu = 4.902800118e12       # lunar gravitational parameter [m^3/s^2]
R_M = 1.738000e6          # lunar reference radius [m]
m0 = 874.4                # initial mass [kg]
Tmax = 2200.0             # maximum thrust [N]
c = 3090.2                # effective exhaust velocity [m/s]
u_min = 0.4               # minimum throttle [-]
u_max = 1.0               # maximum throttle [-]

# Initial and terminal Cartesian states in the Moon-centered inertial frame.
r0 = np.array([1.753000e6, 0.0, 0.0], dtype=float)       # [m]
v0 = np.array([400.0, 800.0, 1100.0], dtype=float)       # [m/s]
rf = np.array([1.66937673e6, 3.3845930e5, 3.4535590e5], dtype=float)  # [m]
vf = np.array([0.0, 0.0, 0.0], dtype=float)              # [m/s]

t0 = 0.0                # initial time [s]
# tf is free and is solved together with the initial costates.

# =========================
# Nondimensionalize
# =========================
# Characteristic units: 1 LU = R_M, 1 TU = sqrt(R_M^3/mu), 1 MU = m0.
LUnit = R_M
TUnit = np.sqrt(LUnit**3 / mu)
MUnit = m0

VUnit = LUnit / TUnit
AUnit = VUnit / TUnit
FUnit = MUnit * AUnit

# Dimensionless state and physical parameters.
x0_nu = np.concatenate((r0 / LUnit, v0 / VUnit)).tolist()
xf_nu = np.concatenate((rf / LUnit, vf / VUnit)).tolist()
t0_nu = t0 / TUnit
m0_nu = m0 / MUnit
mu_nu = mu * TUnit**2 / LUnit**3
Tmax_nu = Tmax / FUnit
c_nu = c / VUnit

# Thrust and exhaust velocity parameters
c1 = Tmax_nu
c2 = c_nu

print(f"LUnit = {LUnit:.9e} m")
print(f"TUnit = {TUnit:.9e} s")
print(f"VUnit = {VUnit:.9e} m/s")
print(f"FUnit = {FUnit:.9e} N")
print(f"mu_nu = {mu_nu:.16e}")
print(f"Tmax_nu = {Tmax_nu:.16e}")
print(f"c_nu = {c_nu:.16e}")
print("x0_nu =", np.asarray(x0_nu))
print("xf_nu =", np.asarray(xf_nu))
print(f"||rf|| / R_M = {np.linalg.norm(rf) / R_M:.16e}")


LUnit = 1.738000000e+06 m
TUnit = 1.034790813e+03 s
VUnit = 1.679566516e+03 m/s
FUnit = 1.419236568e+03 N
mu_nu = 1.0000000000000000e+00
Tmax_nu = 1.5501291679459377e+00
c_nu = 1.8398794991416076e+00
x0_nu = [1.00863061 0.         0.         0.23815669 0.47631338 0.65493089]
xf_nu = [0.96051596 0.19474068 0.1987088  0.         0.         0.        ]
||rf|| / R_M = 1.0000000102770787e+00


In [4]:
import twobody_RV_hamilton_log_lunar

ta = twobody_RV_hamilton_log_lunar.get_integrator()

In [5]:
integration_max_steps = 120
Q_MIN_ALLOWED = -9.0          # eps(q) above 10 eps0 is meaningless; the lower side is clipped at eps_f in the dynamics

# =========================
# Set parameters for heyoka integrator
# =========================
# Cartesian dynamics with logarithmic smoothing

# =========================
# Strict reduced Hamiltonian
# =========================
def strict_reduced_hamiltonian(state, eps, lambda0):
    """Evaluate the strict logarithmic-reduced nondimensional Hamiltonian."""
    
    state = np.asarray(state, dtype=float)
    r = state[0:3]
    v = state[3:6]
    mass = state[6]
    lam_r = state[7:10]
    lam_v = state[10:13]
    lam_m = state[13]

    r_norm = np.linalg.norm(r)
    lam_v_norm = np.linalg.norm(lam_v)
    gravity = -mu_nu * r / r_norm**3

    rho_switch = 1.0 - (c2 * lam_v_norm / mass + lam_m) / lambda0
    u_mid = 0.5 * (u_min + u_max)
    u_half_range = 0.5 * (u_max - u_min)

    H0 = np.dot(lam_r, v) + np.dot(lam_v, gravity)
    H_log = H0 + lambda0 * c1 / c2 * (
        u_mid * rho_switch
        - u_half_range * np.sqrt(rho_switch**2 + 4.0*eps**2)
        + (u_max - u_min) * eps * (1.0 + np.log(
            (np.sqrt(rho_switch**2 + 4.0*eps**2) + 2.0*eps) / eps))
    )
    return float(H_log)

# Residual penalty and terminal-time bounds
BAD_RESIDUAL = 1.0e20
TF_MIN_ALLOWED = t0_nu + 1.0e-10
TF_MAX_ALLOWED = t0_nu + 4.0

def _bad_residual(n):
    # Residual penalty for an invalid trial
    return np.full(n, BAD_RESIDUAL, dtype=float)

def _is_valid_tf(tf_nu):
    # Check the terminal-time bounds
    
    return np.isfinite(tf_nu) and (tf_nu > TF_MIN_ALLOWED) and (tf_nu <= TF_MAX_ALLOWED)

def _set_and_propagate(initial_costates, tf_nu, eps, lambda0):
    """Set the lunar parameters/state and propagate to a free terminal time."""
    # Parameters: [mu, Tmax, c, eps, lambda0, u_min, u_max]
    ta.pars[:] = [mu_nu, c1, c2, eps, lambda0, u_min, u_max]
    ta.time = t0_nu
    ta.state[:7] = x0_nu + [m0_nu]
    ta.state[7:14] = initial_costates
    # Propagate to the trial terminal time tf_nu.
    result = ta.propagate_until(tf_nu, max_steps=integration_max_steps)
    if result[0] != hy.taylor_outcome.time_limit:
        raise RuntimeError('The integration did not reach its terminal time.')

def _lunar_physical_residual(eps, lambda0):
    """Return the 8 free-final-time lunar shooting residuals."""
    # 6 fixed terminal RV conditions + lambda_m(tf)=0 + H_log(tf)=0.
    Hf = strict_reduced_hamiltonian(ta.state[:14], eps, lambda0)
    return np.array([
        ta.state[0] - xf_nu[0],
        ta.state[1] - xf_nu[1],
        ta.state[2] - xf_nu[2],
        ta.state[3] - xf_nu[3],
        ta.state[4] - xf_nu[4],
        ta.state[5] - xf_nu[5],
        ta.state[13],
        Hf,
    ], dtype=float)

# =========================
# Define the shooting functions
# =========================
def shooting(z, eps=1.0e-5, lambda0=1.0, out=False, Time_out=None):
    # z[0:7] = initial costates; z[7] = free terminal time.
    z = np.asarray(z, dtype=float)
    if z.size != 8 or not np.all(np.isfinite(z)):
        return _bad_residual(8)

    initial_costates = z[:7]
    tf_nu = float(z[7])
    if not _is_valid_tf(tf_nu):
        return _bad_residual(8)

    try:
        _set_and_propagate(initial_costates, tf_nu, eps, lambda0)
        retval = _lunar_physical_residual(eps, lambda0)
        if not np.all(np.isfinite(retval)):
            return _bad_residual(8)
    except (ValueError, RuntimeError, OverflowError, FloatingPointError):
        return _bad_residual(8)

    if out:
        print(f"  Final Mass: {ta.state[6] * MUnit:.12f} kg")
        print(f"  Final Time: {tf_nu * TUnit:.12f} s  ({tf_nu:.12f} TU)")
        print(f"  H_log(tf): {retval[-1]:.16e}")

    if Time_out is not None:
        test = Time_out

    return retval

def shooting_augmented(
    z_augmented,
    eps0=1.0,
    epsf=1.0e-5,
    lambda0=1.0,
    out=False,
    Time_out=None,
):
    # z_augmented = [7 initial costates, tf, homotopy variable q].
    # q = (eps0 - eps) / (eps0 - epsf), i.e. eps is LINEAR in q:
    #   q = 0 -> eps = eps0
    #   q = 1 -> eps = epsf
    z_augmented = np.asarray(z_augmented, dtype=float)
    if z_augmented.size != 9 or not np.all(np.isfinite(z_augmented)):
        return _bad_residual(9)

    q = float(z_augmented[8])
    if q < Q_MIN_ALLOWED:
        return _bad_residual(9)

    # The dynamics are integrated with eps never sharper than the target; the homotopy residual uses the raw q
    eps = eps0 + min(q, 1.0) * (epsf - eps0)

    Phi = np.asarray(shooting(z_augmented[:8], eps, lambda0), dtype=float)
    if np.any(np.abs(Phi) >= BAD_RESIDUAL):
        return _bad_residual(9)

    # 8 physical residuals + target-epsilon residual = 9 equations.
    retval = np.concatenate((Phi, [q - 1.0]))

    if out:
        print(f"  Final Mass: {ta.state[6] * MUnit:.12f} kg")
        print(f"  Final Time: {z_augmented[7] * TUnit:.12f} s  ({z_augmented[7]:.12f} TU)")
        print(f"  Final eps : {eps0 + q*(epsf - eps0):.16e}  (integrated with eps = {eps:.16e}, q = {q:.8f})")
        print(f"  H_log(tf): {Phi[-1]:.16e}")

    if Time_out is not None:
        test = Time_out

    return retval

def shooting_augmented_gate(
    z_augmented,
    eps0=1.0,
    epsf=1.0e-5,
    rho0=1.0e-2,
    delta_rho=1.0e-8,
    gate_p=2.0,
    gate_kind="gaussian",
    lambda0=1.0,
    out=False,
    Time_out=None,
):
    # z_augmented = [7 initial costates, tf, homotopy variable q].
    # q = (eps0 - eps) / (eps0 - epsf), i.e. eps is LINEAR in q:
    #   q = 0 -> eps = eps0
    #   q = 1 -> eps = epsf
    z_augmented = np.asarray(z_augmented, dtype=float)
    if z_augmented.size != 9 or not np.all(np.isfinite(z_augmented)):
        return _bad_residual(9)

    q = float(z_augmented[8])
    if q < Q_MIN_ALLOWED:
        return _bad_residual(9)

    # The dynamics are integrated with eps never sharper than the target; the gate equation uses the raw q
    eps = eps0 + min(q, 1.0) * (epsf - eps0)

    Phi = np.asarray(shooting(z_augmented[:8], eps, lambda0), dtype=float)
    if np.any(np.abs(Phi) >= BAD_RESIDUAL):
        return _bad_residual(9)

    # Gate uses the RMS of all 8 lunar physical residuals,
    # including lambda_m(tf) and the strict free-time H_log(tf) residual.
    rho_residual = np.sqrt(delta_rho**2 + np.mean(Phi * Phi))

    if gate_kind == "gaussian":
        w = np.exp(- (rho_residual / rho0) ** gate_p)
    elif gate_kind == "rational":
        w = 1.0 / (1.0 + (rho_residual / rho0) ** gate_p)
    else:
        raise ValueError(f"Unknown gate_kind: {gate_kind}")

    # augmented residual, q measured as the progress in eps, (eps0 - eps)/(eps0 - epsf):
    # when rho is large -> w ~ 0 -> q ~ 0 -> eps ~ eps0
    # when rho is small -> w ~ 1 -> q ~ 1 -> eps ~ epsf
    retval = np.concatenate((Phi, [q - w]))

    if out:
        print(f"  Final Mass: {ta.state[6] * MUnit:.12f} kg")
        print(f"  Final Time: {z_augmented[7] * TUnit:.12f} s  ({z_augmented[7]:.12f} TU)")
        print(
            f"  Final eps : {eps0 + q*(epsf - eps0):.16e}  "
            f"(integrated with eps = {eps:.16e}, q = {q:.8f}, rho = {rho_residual:.3e})"
        )
        print(f"  H_log(tf): {Phi[-1]:.16e}")

    if Time_out is not None:
        test = Time_out

    return retval

# =========================
# Set parameters for minpack
# =========================
import time
from scipy.optimize import root
repeat_num_minpack  = 5000
# repeat_num_minpack  = 1

minpack_tol     = 1.0e-10
minpack_xtol    = 1.0e-10
minpack_maxfev  = 50000
minpack_factor  = 0.1
# Base unknown vector: 7 initial costates + free terminal time.
minpack_diag    = [1.0] * 7 + [1.0]
minpack_eps     = 0.
minpack_error   = 1.0e-7

np.set_printoptions(precision=16, suppress=True)

# =========================
# Convergence is judged by the physical residual at the target smoothing level only
# =========================
def physical_ok(z, eps, lambda0 = 1.0):
    res = np.asarray(shooting(z, eps, lambda0), dtype=float)
    return bool(np.all(np.isfinite(res)) and np.linalg.norm(res) < minpack_error
                and 0.0 < ta.state[6] <= 1.0)

In [6]:

def prepare_timing():
    set_timing_priority()
    if sys.platform != 'darwin':
        return
    started = time.perf_counter()
    deadline = started + 2.0
    cpu = ctypes.c_size_t()
    while True:
        lib.pthread_cpu_number_np(ctypes.byref(cpu))
        if cpu.value in fast_cpus and time.perf_counter()-started >= 0.05:
            return
        if time.perf_counter() > deadline:
            raise RuntimeError('No Super-core scheduling during timing preparation.')

In [7]:
# =========================
# Common random initial guess pool
# =========================
# Generate the common initial guess pool

rng = np.random.default_rng(322)  

# Base unknown vector: [7 initial costates, free terminal time].
tf_nu_min = 0.5
tf_nu_max = TF_MAX_ALLOWED

initial_guess_pool = np.empty((repeat_num_minpack, 8), dtype=float)  
initial_guess_pool[:, :7] = (rng.random((repeat_num_minpack, 7)) - 0.5) * 2.0  
initial_guess_pool[:, 6] = rng.random(repeat_num_minpack)  # lambda_m(0) in (0, 1).
initial_guess_pool[:, 7] = rng.random(repeat_num_minpack) * (tf_nu_max-tf_nu_min) + tf_nu_min  

# Augmented methods use the same base guesses plus q0 = 0.
augmented_initial_guess_pool = np.empty((repeat_num_minpack, 9), dtype=float)  
augmented_initial_guess_pool[:, :8] = initial_guess_pool  
augmented_initial_guess_pool[:, 8] = 0.0  

print(f"Generated {repeat_num_minpack} shared lunar initial guesses.")
print(f"tf guess range: [{initial_guess_pool[:, 7].min():.6f}, {initial_guess_pool[:, 7].max():.6f}] TU")
print(f"lambda_m(0) guess range: [{initial_guess_pool[:, 6].min():.6f}, {initial_guess_pool[:, 6].max():.6f}]")


Generated 5000 shared lunar initial guesses.
tf guess range: [0.500127, 3.999493] TU
lambda_m(0) guess range: [0.000197, 0.999722]


In [8]:
# =========================
# Trial records
# =========================
# The trials of a method are independent. The saved outputs below were obtained by evaluating this
# same loop, one trial per call, in single-thread worker processes; the printed numbers are those of
# a serial run. The timing cells of each method are always measured serially.
def run_records(method, trial_fn):
    records = []
    for i in range(repeat_num_minpack):
        records.append(trial_fn(i))
        if (i + 1) % 500 == 0:
            print(f"{i+1} trials, {sum(r['accepted'] for r in records)} converged", flush=True)
    return records

def report_records(method, records):
    accepted = [r for r in records if r['accepted']]
    print(f"Solved: {len(accepted)} in {len(records)} ({len(accepted)/len(records):.4f})")
    if accepted:
        print(f"Average number of function calls for Minpack: {np.mean([r['nfev'] for r in records]):.0f}")
        print(f"Average number of function calls for convergent solution: {np.mean([r['nfev'] for r in accepted]):.0f}")
        best = max(accepted, key=lambda r: r['mass_kg'])
        print(f"Best final mass: {best['mass_kg']:.6f} kg (trial {best['trial']})")
    records_by_method[method] = records

## Method 1: normal shooting method (eps = 1)

In [9]:
# =========================
# Method 1: normal shooting at eps = 1
# =========================
epsf = 1.0e0
lambda0 = 1.0

def trial_single_eps1(i):
    # Use the shared base initial guess: [lambda(0), tf].
    x0 = initial_guess_pool[i].copy()

    r = root(
        shooting,
        x0,
        args=(epsf, lambda0, False, None),
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )

    ok = bool(np.all(np.isfinite(r.x)) and physical_ok(r.x, epsf, lambda0))
    mass = float(ta.state[6]*MUnit) if ok else None
    return dict(trial=int(i), accepted=ok, nfev=int(r.nfev), status=int(r.status),
                residual_norm=float(np.linalg.norm(r.fun)), eps=epsf, mass_kg=mass, x=r.x.tolist())

In [10]:
records = run_records('Single_eps1', trial_single_eps1)
report_records('Single_eps1', records)

500 trials, 105 converged
1000 trials, 220 converged
1500 trials, 332 converged
2000 trials, 455 converged
2500 trials, 574 converged
3000 trials, 689 converged
3500 trials, 810 converged
4000 trials, 935 converged
4500 trials, 1050 converged
5000 trials, 1170 converged
Solved: 1170 in 5000 (0.2340)
Average number of function calls for Minpack: 42
Average number of function calls for convergent solution: 75
Best final mass: 495.067494 kg (trial 3900)


In [11]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['Single_eps1']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()

In [12]:
%%timeit -n 1 -r 7 -v timing_all prepare_timing()
for i in timing_ids:
    sample_cpu()
    trial_single_eps1(i)

1.09 s ± 13.1 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [13]:
%%timeit -n 1 -r 7 -v timing_successful prepare_timing()
for i in successful_ids:
    sample_cpu()
    trial_single_eps1(i)

142 ms ± 277 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [14]:
%%timeit -n 1 -r 7 -v timing_failed prepare_timing()
for i in failed_ids:
    sample_cpu()
    trial_single_eps1(i)

934 ms ± 14.7 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [15]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['Single_eps1'] = dict(
    method='Single shooting, epsilon=1', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['Single_eps1'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['Single_eps1']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)

## Method 1: normal shooting method (eps = 1e-5)

In [16]:
# =========================
# Method 1: normal shooting at eps = 1e-5
# =========================
epsf = 1.0e-5
lambda0 = 1.0

def trial_single_epsf(i):
    # Use the shared base initial guess: [lambda(0), tf].
    x0 = initial_guess_pool[i].copy()

    r = root(
        shooting,
        x0,
        args=(epsf, lambda0, False, None),
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )

    ok = bool(np.all(np.isfinite(r.x)) and physical_ok(r.x, epsf, lambda0))
    mass = float(ta.state[6]*MUnit) if ok else None
    return dict(trial=int(i), accepted=ok, nfev=int(r.nfev), status=int(r.status),
                residual_norm=float(np.linalg.norm(r.fun)), eps=epsf, mass_kg=mass, x=r.x.tolist())

In [17]:
records = run_records('Single_epsf', trial_single_epsf)
report_records('Single_epsf', records)

500 trials, 71 converged
1000 trials, 148 converged
1500 trials, 217 converged
2000 trials, 305 converged
2500 trials, 381 converged
3000 trials, 459 converged
3500 trials, 536 converged
4000 trials, 624 converged
4500 trials, 707 converged
5000 trials, 788 converged
Solved: 788 in 5000 (0.1576)
Average number of function calls for Minpack: 44
Average number of function calls for convergent solution: 106
Best final mass: 510.407998 kg (trial 1609)


In [18]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['Single_epsf']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()

In [19]:
%%timeit -n 1 -r 7 -v timing_all prepare_timing()
for i in timing_ids:
    sample_cpu()
    trial_single_epsf(i)

1.39 s ± 18.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [20]:
%%timeit -n 1 -r 7 -v timing_successful prepare_timing()
for i in successful_ids:
    sample_cpu()
    trial_single_epsf(i)

336 ms ± 5.31 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [21]:
%%timeit -n 1 -r 7 -v timing_failed prepare_timing()
for i in failed_ids:
    sample_cpu()
    trial_single_epsf(i)

1.03 s ± 2.23 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [22]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['Single_epsf'] = dict(
    method='Single shooting, epsilon=1e-5', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['Single_epsf'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['Single_epsf']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)

## Method 2: consecutive iterations on the smoothing parameter

In [23]:
# =========================
# Method 2: consecutive iterations on the smoothing parameter
# =========================
# Continuation in eps: eps0 -> epsf
eps0 = 1.0e0
epsf = 1.0e-5
lambda0 = 1.0

# continuation/backtracking parameters (eps_next = eps_curr * d)
d0 = 0.1                 # initial step ratio (aggressive)
d_shrink = 0.8           # backtracking factor (used to INCREASE d so eps_next is closer to eps_curr)
d_recover = 0.8          # gentle recovery factor (after success, gradually DECREASE d back toward d0)
d_max = 0.999            # upper bound of d (avoid d==1 causing stagnation)
max_backtrack = 20       # max backtracking tries per step
max_total_steps = 300    # max accepted eps steps per continuation path (avoid infinite loops)

# wrap a single root() call to unify nfev and the success criterion
def solve_for_eps(x_init, eps_val):
    r = root(
        shooting,
        x_init,
        args=(eps_val, lambda0, False, None),
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )

    ok = bool(np.all(np.isfinite(r.x)) and physical_ok(r.x, eps_val, lambda0))
    return r, ok

def trial_stepwise(i):
    # Use the shared base initial guess: [lambda(0), tf].
    x0 = initial_guess_pool[i].copy()

    # continuation initialization
    eps_curr = eps0
    d = d0
    x_guess = x0.copy()
    nfev = 0
    solved_this_outer = False

    # ============================================================
    # 0) Solve the easiest problem at eps0 ONCE (outside inner loop)
    # ============================================================
    r, ok0 = solve_for_eps(x_guess, eps_curr)
    nfev += r.nfev

    # ============================================================
    # 1) Continuation: advance eps until epsf (accepted eps only)
    # ============================================================
    if ok0:
        x_guess = r.x.copy()
        for step in range(max_total_steps):

            # Stop if target reached
            if eps_curr <= epsf:
                solved_this_outer = True
                break

            # Backtracking search for a feasible eps_next = eps_curr * d
            advanced = False
            for backtrack in range(max_backtrack + 1):
                eps_next = max(eps_curr * d, epsf)

                # Numerical safety: if eps_next does not change, force epsf
                if eps_next == eps_curr:
                    eps_next = epsf

                r, ok_next = solve_for_eps(x_guess, eps_next)
                nfev += r.nfev

                if ok_next:
                    # Accept eps_next
                    eps_curr = eps_next
                    x_guess = r.x.copy()
                    # gentle recovery of d (after success, gradually decrease d back toward d0)
                    d = max(d0, d * d_recover)
                    advanced = True
                    break
                else:
                    # Increase d to bring eps_next closer to eps_curr
                    d = min(d_max, d / d_shrink)

            if not advanced:
                # Too many backtracks and still cannot advance -> fail this random initial guess
                break

    solved = bool(solved_this_outer and physical_ok(x_guess, epsf, lambda0))
    mass = float(ta.state[6]*MUnit) if solved else None
    return dict(trial=int(i), accepted=solved, nfev=int(nfev), status=int(r.status),
                residual_norm=float(np.linalg.norm(r.fun)), eps=float(eps_curr), mass_kg=mass, x=x_guess.tolist())

In [24]:
records = run_records('Stepwise', trial_stepwise)
report_records('Stepwise', records)

500 trials, 105 converged
1000 trials, 220 converged
1500 trials, 332 converged
2000 trials, 455 converged
2500 trials, 574 converged
3000 trials, 689 converged
3500 trials, 810 converged
4000 trials, 935 converged
4500 trials, 1050 converged
5000 trials, 1170 converged
Solved: 1170 in 5000 (0.2340)
Average number of function calls for Minpack: 70
Average number of function calls for convergent solution: 197
Best final mass: 510.407998 kg (trial 2317)


In [25]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['Stepwise']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()

In [26]:
%%timeit -n 1 -r 7 -v timing_all prepare_timing()
for i in timing_ids:
    sample_cpu()
    trial_stepwise(i)

1.73 s ± 1.27 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [27]:
%%timeit -n 1 -r 7 -v timing_successful prepare_timing()
for i in successful_ids:
    sample_cpu()
    trial_stepwise(i)

811 ms ± 1.83 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [28]:
%%timeit -n 1 -r 7 -v timing_failed prepare_timing()
for i in failed_ids:
    sample_cpu()
    trial_stepwise(i)

938 ms ± 9.73 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [29]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['Stepwise'] = dict(
    method='Stepwise', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['Stepwise'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['Stepwise']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)

## Method 3: augmented shooting method

In [30]:
# =========================
# Method 3: augmented shooting, ungated (q = 1)
# =========================
eps0 = 1.0e0
epsf = 1.0e-5
lambda0 = 1.0

def trial_ungated(i):
    # Shared base initial guess plus the homotopy variable: [lambda(0), tf, q].
    x0 = augmented_initial_guess_pool[i].copy()

    r = root(
        shooting_augmented,
        x0,
        args=(eps0, epsf, lambda0, False, None),
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag + [1.],
            'eps': minpack_eps,
        }
    )

    ok = bool(np.all(np.isfinite(r.x)) and physical_ok(r.x[:8], epsf, lambda0))
    mass = float(ta.state[6]*MUnit) if ok else None
    return dict(trial=int(i), accepted=ok, nfev=int(r.nfev), status=int(r.status),
                residual_norm=float(np.linalg.norm(r.fun)), eps=float(eps0 + r.x[8]*(epsf - eps0)),
                q=float(r.x[8]), mass_kg=mass, x=r.x.tolist())

In [31]:
records = run_records('Ungated', trial_ungated)
report_records('Ungated', records)

500 trials, 108 converged
1000 trials, 222 converged
1500 trials, 329 converged
2000 trials, 453 converged
2500 trials, 562 converged
3000 trials, 675 converged
3500 trials, 792 converged
4000 trials, 914 converged
4500 trials, 1022 converged
5000 trials, 1139 converged
Solved: 1139 in 5000 (0.2278)
Average number of function calls for Minpack: 48
Average number of function calls for convergent solution: 96
Best final mass: 510.407998 kg (trial 2602)


In [32]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['Ungated']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()

In [33]:
%%timeit -n 1 -r 7 -v timing_all prepare_timing()
for i in timing_ids:
    sample_cpu()
    trial_ungated(i)

1.44 s ± 10.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [34]:
%%timeit -n 1 -r 7 -v timing_successful prepare_timing()
for i in successful_ids:
    sample_cpu()
    trial_ungated(i)

414 ms ± 3.51 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [35]:
%%timeit -n 1 -r 7 -v timing_failed prepare_timing()
for i in failed_ids:
    sample_cpu()
    trial_ungated(i)

1.02 s ± 5.81 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [36]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['Ungated'] = dict(
    method='Ungated', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['Ungated'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['Ungated']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)

## Method 3: augmented shooting method with exponential gate (p = 1)

In [37]:
# =========================
# Method 3: augmented shooting with the residual gate (exponential, p = 1)
# =========================
eps0 = 1.0e0
epsf = 1.0e-5

# parameters for the augmented residual, selected in Case 1 and transferred without retuning
gate_option = 'gaussian'
gate_p = 1.0
rho0 = 0.316796

# [other parameters]:
fd_eps = 0.
delta_rho = 1.0e-8

# [general settings]:
lambda0 = 1.0

def trial_gate_exp_p1(i):
    # Shared base initial guess plus the homotopy variable: [lambda(0), tf, q].
    x0 = augmented_initial_guess_pool[i].copy()

    r = root(
        shooting_augmented_gate,
        x0,
        # pass eps0, epsf, and gate parameters
        args=(eps0, epsf, rho0, delta_rho, gate_p, gate_option, lambda0, False, None),
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag + [1.],
            # 'eps': minpack_eps,
            'eps': fd_eps,
        }
    )

    ok = bool(np.all(np.isfinite(r.x)) and physical_ok(r.x[:8], epsf, lambda0))
    mass = float(ta.state[6]*MUnit) if ok else None
    return dict(trial=int(i), accepted=ok, nfev=int(r.nfev), status=int(r.status),
                residual_norm=float(np.linalg.norm(r.fun)), eps=float(eps0 + r.x[8]*(epsf - eps0)),
                q=float(r.x[8]), mass_kg=mass, x=r.x.tolist())

In [38]:
records = run_records('Gated_Exp_p1', trial_gate_exp_p1)
report_records('Gated_Exp_p1', records)

500 trials, 0 converged
1000 trials, 1 converged
1500 trials, 1 converged
2000 trials, 2 converged
2500 trials, 2 converged
3000 trials, 6 converged
3500 trials, 7 converged
4000 trials, 8 converged
4500 trials, 9 converged
5000 trials, 9 converged
Solved: 9 in 5000 (0.0018)
Average number of function calls for Minpack: 65
Average number of function calls for convergent solution: 177
Best final mass: 510.408001 kg (trial 2992)


In [39]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['Gated_Exp_p1']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()

In [40]:
%%timeit -n 1 -r 7 -v timing_all prepare_timing()
for i in timing_ids:
    sample_cpu()
    trial_gate_exp_p1(i)

1.82 s ± 17.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [41]:
%%timeit -n 1 -r 7 -v timing_successful prepare_timing()
for i in successful_ids:
    sample_cpu()
    trial_gate_exp_p1(i)

464 ns ± 433 ns per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [42]:
%%timeit -n 1 -r 7 -v timing_failed prepare_timing()
for i in failed_ids:
    sample_cpu()
    trial_gate_exp_p1(i)

1.84 s ± 26.2 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [43]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['Gated_Exp_p1'] = dict(
    method='Augmented, W_exp, p=1', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['Gated_Exp_p1'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['Gated_Exp_p1']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)

## Method 3: augmented shooting method with exponential gate (p = 2)

In [44]:
# =========================
# Method 3: augmented shooting with the residual gate (exponential, p = 2)
# =========================
eps0 = 1.0e0
epsf = 1.0e-5

# parameters for the augmented residual, selected in Case 1 and transferred without retuning
gate_option = 'gaussian'
gate_p = 2.0
rho0 = 0.737159

# [other parameters]:
fd_eps = 0.
delta_rho = 1.0e-8

# [general settings]:
lambda0 = 1.0

def trial_gate_exp_p2(i):
    # Shared base initial guess plus the homotopy variable: [lambda(0), tf, q].
    x0 = augmented_initial_guess_pool[i].copy()

    r = root(
        shooting_augmented_gate,
        x0,
        # pass eps0, epsf, and gate parameters
        args=(eps0, epsf, rho0, delta_rho, gate_p, gate_option, lambda0, False, None),
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag + [1.],
            # 'eps': minpack_eps,
            'eps': fd_eps,
        }
    )

    ok = bool(np.all(np.isfinite(r.x)) and physical_ok(r.x[:8], epsf, lambda0))
    mass = float(ta.state[6]*MUnit) if ok else None
    return dict(trial=int(i), accepted=ok, nfev=int(r.nfev), status=int(r.status),
                residual_norm=float(np.linalg.norm(r.fun)), eps=float(eps0 + r.x[8]*(epsf - eps0)),
                q=float(r.x[8]), mass_kg=mass, x=r.x.tolist())

In [45]:
records = run_records('Gated_Exp_p2', trial_gate_exp_p2)
report_records('Gated_Exp_p2', records)

500 trials, 106 converged
1000 trials, 228 converged
1500 trials, 338 converged
2000 trials, 463 converged
2500 trials, 570 converged
3000 trials, 684 converged
3500 trials, 801 converged
4000 trials, 922 converged
4500 trials, 1040 converged
5000 trials, 1160 converged
Solved: 1160 in 5000 (0.2320)
Average number of function calls for Minpack: 49
Average number of function calls for convergent solution: 98
Best final mass: 510.407998 kg (trial 2243)


In [46]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['Gated_Exp_p2']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()

In [47]:
%%timeit -n 1 -r 7 -v timing_all prepare_timing()
for i in timing_ids:
    sample_cpu()
    trial_gate_exp_p2(i)

1.44 s ± 1.66 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [48]:
%%timeit -n 1 -r 7 -v timing_successful prepare_timing()
for i in successful_ids:
    sample_cpu()
    trial_gate_exp_p2(i)

420 ms ± 1.06 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [49]:
%%timeit -n 1 -r 7 -v timing_failed prepare_timing()
for i in failed_ids:
    sample_cpu()
    trial_gate_exp_p2(i)

1.02 s ± 4.09 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [50]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['Gated_Exp_p2'] = dict(
    method='Augmented, W_exp, p=2', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['Gated_Exp_p2'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['Gated_Exp_p2']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)

## Method 3: augmented shooting method with exponential gate (p = 4)

In [51]:
# =========================
# Method 3: augmented shooting with the residual gate (exponential, p = 4)
# =========================
eps0 = 1.0e0
epsf = 1.0e-5

# parameters for the augmented residual, selected in Case 1 and transferred without retuning
gate_option = 'gaussian'
gate_p = 4.0
rho0 = 0.293384

# [other parameters]:
fd_eps = 0.
delta_rho = 1.0e-8

# [general settings]:
lambda0 = 1.0

def trial_gate_exp_p4(i):
    # Shared base initial guess plus the homotopy variable: [lambda(0), tf, q].
    x0 = augmented_initial_guess_pool[i].copy()

    r = root(
        shooting_augmented_gate,
        x0,
        # pass eps0, epsf, and gate parameters
        args=(eps0, epsf, rho0, delta_rho, gate_p, gate_option, lambda0, False, None),
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag + [1.],
            # 'eps': minpack_eps,
            'eps': fd_eps,
        }
    )

    ok = bool(np.all(np.isfinite(r.x)) and physical_ok(r.x[:8], epsf, lambda0))
    mass = float(ta.state[6]*MUnit) if ok else None
    return dict(trial=int(i), accepted=ok, nfev=int(r.nfev), status=int(r.status),
                residual_norm=float(np.linalg.norm(r.fun)), eps=float(eps0 + r.x[8]*(epsf - eps0)),
                q=float(r.x[8]), mass_kg=mass, x=r.x.tolist())

In [52]:
records = run_records('Gated_Exp_p4', trial_gate_exp_p4)
report_records('Gated_Exp_p4', records)

500 trials, 98 converged
1000 trials, 208 converged
1500 trials, 314 converged
2000 trials, 434 converged
2500 trials, 538 converged
3000 trials, 641 converged
3500 trials, 761 converged
4000 trials, 880 converged
4500 trials, 989 converged
5000 trials, 1102 converged
Solved: 1102 in 5000 (0.2204)
Average number of function calls for Minpack: 50
Average number of function calls for convergent solution: 100
Best final mass: 510.407998 kg (trial 2333)


In [53]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['Gated_Exp_p4']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()

In [54]:
%%timeit -n 1 -r 7 -v timing_all prepare_timing()
for i in timing_ids:
    sample_cpu()
    trial_gate_exp_p4(i)

1.44 s ± 11.5 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [55]:
%%timeit -n 1 -r 7 -v timing_successful prepare_timing()
for i in successful_ids:
    sample_cpu()
    trial_gate_exp_p4(i)

390 ms ± 1.59 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [56]:
%%timeit -n 1 -r 7 -v timing_failed prepare_timing()
for i in failed_ids:
    sample_cpu()
    trial_gate_exp_p4(i)

1.04 s ± 1.35 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [57]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['Gated_Exp_p4'] = dict(
    method='Augmented, W_exp, p=4', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['Gated_Exp_p4'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['Gated_Exp_p4']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)

## Method 3: augmented shooting method with rational gate (p = 1)

In [58]:
# =========================
# Method 3: augmented shooting with the residual gate (rational, p = 1)
# =========================
eps0 = 1.0e0
epsf = 1.0e-5

# parameters for the augmented residual, selected in Case 1 and transferred without retuning
gate_option = 'rational'
gate_p = 1.0
rho0 = 0.233027

# [other parameters]:
fd_eps = 0.
delta_rho = 1.0e-8

# [general settings]:
lambda0 = 1.0

def trial_gate_rational_p1(i):
    # Shared base initial guess plus the homotopy variable: [lambda(0), tf, q].
    x0 = augmented_initial_guess_pool[i].copy()

    r = root(
        shooting_augmented_gate,
        x0,
        # pass eps0, epsf, and gate parameters
        args=(eps0, epsf, rho0, delta_rho, gate_p, gate_option, lambda0, False, None),
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag + [1.],
            # 'eps': minpack_eps,
            'eps': fd_eps,
        }
    )

    ok = bool(np.all(np.isfinite(r.x)) and physical_ok(r.x[:8], epsf, lambda0))
    mass = float(ta.state[6]*MUnit) if ok else None
    return dict(trial=int(i), accepted=ok, nfev=int(r.nfev), status=int(r.status),
                residual_norm=float(np.linalg.norm(r.fun)), eps=float(eps0 + r.x[8]*(epsf - eps0)),
                q=float(r.x[8]), mass_kg=mass, x=r.x.tolist())

In [59]:
records = run_records('Gated_Rational_p1', trial_gate_rational_p1)
report_records('Gated_Rational_p1', records)

500 trials, 3 converged
1000 trials, 6 converged
1500 trials, 6 converged
2000 trials, 6 converged
2500 trials, 7 converged
3000 trials, 9 converged
3500 trials, 11 converged
4000 trials, 12 converged
4500 trials, 13 converged
5000 trials, 15 converged
Solved: 15 in 5000 (0.0030)
Average number of function calls for Minpack: 67
Average number of function calls for convergent solution: 210
Best final mass: 510.408004 kg (trial 2940)


In [60]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['Gated_Rational_p1']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()

In [61]:
%%timeit -n 1 -r 7 -v timing_all prepare_timing()
for i in timing_ids:
    sample_cpu()
    trial_gate_rational_p1(i)

1.87 s ± 5.01 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [62]:
%%timeit -n 1 -r 7 -v timing_successful prepare_timing()
for i in successful_ids:
    sample_cpu()
    trial_gate_rational_p1(i)

32.8 ms ± 167 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [63]:
%%timeit -n 1 -r 7 -v timing_failed prepare_timing()
for i in failed_ids:
    sample_cpu()
    trial_gate_rational_p1(i)

1.84 s ± 6.89 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [64]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['Gated_Rational_p1'] = dict(
    method='Augmented, W_rat, p=1', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['Gated_Rational_p1'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['Gated_Rational_p1']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)

## Method 3: augmented shooting method with rational gate (p = 2)

In [65]:
# =========================
# Method 3: augmented shooting with the residual gate (rational, p = 2)
# =========================
eps0 = 1.0e0
epsf = 1.0e-5

# parameters for the augmented residual, selected in Case 1 and transferred without retuning
gate_option = 'rational'
gate_p = 2.0
rho0 = 0.185086

# [other parameters]:
fd_eps = 0.
delta_rho = 1.0e-8

# [general settings]:
lambda0 = 1.0

def trial_gate_rational_p2(i):
    # Shared base initial guess plus the homotopy variable: [lambda(0), tf, q].
    x0 = augmented_initial_guess_pool[i].copy()

    r = root(
        shooting_augmented_gate,
        x0,
        # pass eps0, epsf, and gate parameters
        args=(eps0, epsf, rho0, delta_rho, gate_p, gate_option, lambda0, False, None),
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag + [1.],
            # 'eps': minpack_eps,
            'eps': fd_eps,
        }
    )

    ok = bool(np.all(np.isfinite(r.x)) and physical_ok(r.x[:8], epsf, lambda0))
    mass = float(ta.state[6]*MUnit) if ok else None
    return dict(trial=int(i), accepted=ok, nfev=int(r.nfev), status=int(r.status),
                residual_norm=float(np.linalg.norm(r.fun)), eps=float(eps0 + r.x[8]*(epsf - eps0)),
                q=float(r.x[8]), mass_kg=mass, x=r.x.tolist())

In [66]:
records = run_records('Gated_Rational_p2', trial_gate_rational_p2)
report_records('Gated_Rational_p2', records)

500 trials, 102 converged
1000 trials, 220 converged
1500 trials, 333 converged
2000 trials, 458 converged
2500 trials, 571 converged
3000 trials, 684 converged
3500 trials, 804 converged
4000 trials, 925 converged
4500 trials, 1036 converged
5000 trials, 1157 converged
Solved: 1157 in 5000 (0.2314)
Average number of function calls for Minpack: 53
Average number of function calls for convergent solution: 117
Best final mass: 510.407998 kg (trial 3304)


In [67]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['Gated_Rational_p2']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()

In [68]:
%%timeit -n 1 -r 7 -v timing_all prepare_timing()
for i in timing_ids:
    sample_cpu()
    trial_gate_rational_p2(i)

1.5 s ± 8.58 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [69]:
%%timeit -n 1 -r 7 -v timing_successful prepare_timing()
for i in successful_ids:
    sample_cpu()
    trial_gate_rational_p2(i)

472 ms ± 675 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [70]:
%%timeit -n 1 -r 7 -v timing_failed prepare_timing()
for i in failed_ids:
    sample_cpu()
    trial_gate_rational_p2(i)

1.04 s ± 23.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [71]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['Gated_Rational_p2'] = dict(
    method='Augmented, W_rat, p=2', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['Gated_Rational_p2'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['Gated_Rational_p2']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)

## Method 3: augmented shooting method with rational gate (p = 4)

In [72]:
# =========================
# Method 3: augmented shooting with the residual gate (rational, p = 4)
# =========================
eps0 = 1.0e0
epsf = 1.0e-5

# parameters for the augmented residual, selected in Case 1 and transferred without retuning
gate_option = 'rational'
gate_p = 4.0
rho0 = 0.171408

# [other parameters]:
fd_eps = 0.
delta_rho = 1.0e-8

# [general settings]:
lambda0 = 1.0

def trial_gate_rational_p4(i):
    # Shared base initial guess plus the homotopy variable: [lambda(0), tf, q].
    x0 = augmented_initial_guess_pool[i].copy()

    r = root(
        shooting_augmented_gate,
        x0,
        # pass eps0, epsf, and gate parameters
        args=(eps0, epsf, rho0, delta_rho, gate_p, gate_option, lambda0, False, None),
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag + [1.],
            # 'eps': minpack_eps,
            'eps': fd_eps,
        }
    )

    ok = bool(np.all(np.isfinite(r.x)) and physical_ok(r.x[:8], epsf, lambda0))
    mass = float(ta.state[6]*MUnit) if ok else None
    return dict(trial=int(i), accepted=ok, nfev=int(r.nfev), status=int(r.status),
                residual_norm=float(np.linalg.norm(r.fun)), eps=float(eps0 + r.x[8]*(epsf - eps0)),
                q=float(r.x[8]), mass_kg=mass, x=r.x.tolist())

In [73]:
records = run_records('Gated_Rational_p4', trial_gate_rational_p4)
report_records('Gated_Rational_p4', records)

500 trials, 102 converged
1000 trials, 216 converged
1500 trials, 322 converged
2000 trials, 447 converged
2500 trials, 558 converged
3000 trials, 670 converged
3500 trials, 789 converged
4000 trials, 910 converged
4500 trials, 1024 converged
5000 trials, 1145 converged
Solved: 1145 in 5000 (0.2290)
Average number of function calls for Minpack: 51
Average number of function calls for convergent solution: 107
Best final mass: 510.407999 kg (trial 2309)


In [74]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['Gated_Rational_p4']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()

In [75]:
%%timeit -n 1 -r 7 -v timing_all prepare_timing()
for i in timing_ids:
    sample_cpu()
    trial_gate_rational_p4(i)

1.47 s ± 19.2 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [76]:
%%timeit -n 1 -r 7 -v timing_successful prepare_timing()
for i in successful_ids:
    sample_cpu()
    trial_gate_rational_p4(i)

419 ms ± 6.02 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [77]:
%%timeit -n 1 -r 7 -v timing_failed prepare_timing()
for i in failed_ids:
    sample_cpu()
    trial_gate_rational_p4(i)

1.05 s ± 9.03 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [78]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['Gated_Rational_p4'] = dict(
    method='Augmented, W_rat, p=4', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['Gated_Rational_p4'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['Gated_Rational_p4']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)

## Summary

In [79]:

headings = ['Method', 'Success / 5000', 'Convergence [%]', 'Expected first [ms]',
            'Successful mean [ms]', 'Failed mean [ms]', 'All-trial mean [ms]', 'Nfev (all)', 'Nfev (converged)']
table = ['| '+' | '.join(headings)+' |', '| '+' | '.join(['---']*len(headings))+' |']
for value in results_by_method.values():
    values = [value['method'], value['accepted'], f"{100.*value['accepted']/value['trials']:.2f}"]
    values += [f"{value[k]:.3f}" for k in ('first_ms','successful_ms','failed_ms','all_ms')]
    values += [f"{value['nfev_all']:.1f}", f"{value['nfev_successful']:.1f}"]
    table.append('| '+' | '.join(map(str,values))+' |')
display(Markdown('\n'.join(table)))

| Method | Success / 5000 | Convergence [%] | Expected first [ms] | Successful mean [ms] | Failed mean [ms] | All-trial mean [ms] | Nfev (all) | Nfev (converged) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Single shooting, epsilon=1 | 1170 | 23.40 | 9.300 | 1.351 | 2.365 | 2.176 | 41.6 | 74.9 |
| Single shooting, epsilon=1e-5 | 788 | 15.76 | 17.639 | 4.729 | 2.410 | 2.780 | 44.2 | 105.5 |
| Stepwise | 1170 | 23.40 | 14.820 | 7.719 | 2.374 | 3.468 | 70.2 | 196.9 |
| Ungated | 1139 | 22.78 | 12.630 | 3.836 | 2.604 | 2.877 | 48.5 | 96.3 |
| Augmented, W_exp, p=1 | 9 | 0.18 | 2025.733 | nan | 3.672 | 3.646 | 64.8 | 176.9 |
| Augmented, W_exp, p=2 | 1160 | 23.20 | 12.428 | 3.965 | 2.594 | 2.883 | 49.1 | 98.5 |
| Augmented, W_exp, p=4 | 1102 | 22.04 | 13.058 | 3.981 | 2.581 | 2.878 | 49.5 | 99.8 |
| Augmented, W_rat, p=1 | 15 | 0.30 | 1244.866 | 10.927 | 3.705 | 3.735 | 67.3 | 209.7 |
| Augmented, W_rat, p=2 | 1157 | 23.14 | 12.981 | 4.630 | 2.613 | 3.004 | 53.1 | 116.7 |
| Augmented, W_rat, p=4 | 1145 | 22.90 | 12.812 | 4.111 | 2.633 | 2.934 | 50.8 | 107.4 |